<a href="https://colab.research.google.com/github/mostafizcse007/Machine-Learning/blob/main/Week%206/xgboost_part_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [76]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import optuna

from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split,KFold, cross_val_score
from sklearn.metrics import r2_score
from xgboost import XGBRegressor,callback
import xgboost

In [77]:
train = pd.read_csv('/content/train.csv')
test = pd.read_csv('/content/test.csv')

In [78]:
ids = test['id']

In [79]:
train.drop(columns=['id'],inplace = True)
test.drop(columns=['id'],inplace = True)

In [80]:
train = pd.get_dummies(data=train,columns=['Sex'],drop_first=True,dtype=int)
test = pd.get_dummies(data=test,columns=['Sex'],drop_first=True,dtype=int)

In [81]:
train.head()

,Length,Diameter,Height,Weight,Shucked Weight,Viscera Weight,Shell Weight,Age,Sex_I,Sex_M
0,1.6000,1.2500,0.5750,43.700754,15.025235,10.262519,13.324265,17.0,0,0
1,1.5750,1.1625,0.3750,32.417653,12.927372,6.208540,8.930093,20.0,0,1
2,1.6125,1.2750,0.4375,34.373769,14.784264,9.142714,10.772810,10.0,0,0
3,1.2250,0.9875,0.3250,18.455524,8.037083,3.926406,4.819415,9.0,0,1
4,1.3750,1.0250,0.3875,22.736299,9.879801,5.513978,6.945627,10.0,1,0


In [82]:
X,y = train.drop(columns=['Age']),train['Age']

In [83]:
X_train_full,X_test,y_train_full,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [84]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, test_size=0.25, random_state=42
)

In [85]:
def objective(trial):
  params = {
      "objective":"reg:squarederror",
      "eval_metric":"mae",
      "tree_method":"hist",
      "random_state":42,
      "verbosity":0,
      "max_depth":trial.suggest_int("max_depth",3,15,step=1),
      "n_estimators":trial.suggest_int("n_estimators",100,5000,step=100),
      "learning_rate":trial.suggest_float("learning_rate",1e-3,0.3,log=True),
      "min_child_weight":trial.suggest_int("min_child_weight",1,20),
      "gamma":trial.suggest_float("gamma",1.0,10.0),
      "subsample":trial.suggest_float("subsample",0.5,1.0),
      "colsample_bytree":trial.suggest_float("colsample_bytree",0.5,1.0),
      "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
      "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True)
  }

  es = xgboost.callback.EarlyStopping(
    rounds=50,
    save_best=True
 )
  clf = XGBRegressor(tree_method="hist", device="cuda", callbacks=[es])
  clf.fit(X_train,y_train,eval_set=[(X_valid,y_valid)])
  y_pred = clf.predict(X_valid)
  score = r2_score(y_valid,y_pred)
  return score

In [86]:
study = optuna.create_study(direction='maximize')
study.optimize(objective,n_trials=50)

[I 2026-03-20 15:15:07,890] A new study created in memory with name: no-name-b2f15553-c88f-4561-94e4-ee4ba9d072bb


[0]	validation_0-rmse:2.60587
[1]	validation_0-rmse:2.31108
[2]	validation_0-rmse:2.15219
[3]	validation_0-rmse:2.07163
[4]	validation_0-rmse:2.02273
[5]	validation_0-rmse:1.99900
[6]	validation_0-rmse:1.99001
[7]	validation_0-rmse:1.98057
[8]	validation_0-rmse:1.98459
[9]	validation_0-rmse:1.98631
[10]	validation_0-rmse:1.98279
[11]	validation_0-rmse:1.98039
[12]	validation_0-rmse:1.98315
[13]	validation_0-rmse:1.98357
[14]	validation_0-rmse:1.98491
[15]	validation_0-rmse:1.98553
[16]	validation_0-rmse:1.98514
[17]	validation_0-rmse:1.98524
[18]	validation_0-rmse:1.98522
[19]	validation_0-rmse:1.98488
[20]	validation_0-rmse:1.98572
[21]	validation_0-rmse:1.98186
[22]	validation_0-rmse:1.98311
[23]	validation_0-rmse:1.98358
[24]	validation_0-rmse:1.98593
[25]	validation_0-rmse:1.98598
[26]	validation_0-rmse:1.98688
[27]	validation_0-rmse:1.98729
[28]	validation_0-rmse:1.98636
[29]	validation_0-rmse:1.98904
[30]	validation_0-rmse:1.99021
[31]	validation_0-rmse:1.99089
[32]	validation_0-

[I 2026-03-20 15:15:08,167] Trial 0 finished with value: 0.5955685956657008 and parameters: {'max_depth': 11, 'n_estimators': 4700, 'learning_rate': 0.05532932798000539, 'min_child_weight': 15, 'gamma': 1.356098712183698, 'subsample': 0.581094517772616, 'colsample_bytree': 0.7659330310384236, 'reg_alpha': 1.351853793308209, 'reg_lambda': 0.36773855146634393}. Best is trial 0 with value: 0.5955685956657008.


[0]	validation_0-rmse:2.60587
[1]	validation_0-rmse:2.31108
[2]	validation_0-rmse:2.15219
[3]	validation_0-rmse:2.07163
[4]	validation_0-rmse:2.02273
[5]	validation_0-rmse:1.99900
[6]	validation_0-rmse:1.99001
[7]	validation_0-rmse:1.98057
[8]	validation_0-rmse:1.98459
[9]	validation_0-rmse:1.98631
[10]	validation_0-rmse:1.98279
[11]	validation_0-rmse:1.98039
[12]	validation_0-rmse:1.98315
[13]	validation_0-rmse:1.98357
[14]	validation_0-rmse:1.98491
[15]	validation_0-rmse:1.98553
[16]	validation_0-rmse:1.98514
[17]	validation_0-rmse:1.98524
[18]	validation_0-rmse:1.98522
[19]	validation_0-rmse:1.98488
[20]	validation_0-rmse:1.98572
[21]	validation_0-rmse:1.98186
[22]	validation_0-rmse:1.98311
[23]	validation_0-rmse:1.98358
[24]	validation_0-rmse:1.98593
[25]	validation_0-rmse:1.98598
[26]	validation_0-rmse:1.98688
[27]	validation_0-rmse:1.98729
[28]	validation_0-rmse:1.98636
[29]	validation_0-rmse:1.98904
[30]	validation_0-rmse:1.99021
[31]	validation_0-rmse:1.99089
[32]	validation_0-

[I 2026-03-20 15:15:08,441] Trial 1 finished with value: 0.5955685956657008 and parameters: {'max_depth': 14, 'n_estimators': 3800, 'learning_rate': 0.12132613069918431, 'min_child_weight': 15, 'gamma': 8.012522557395837, 'subsample': 0.580493013585357, 'colsample_bytree': 0.5448908179173628, 'reg_alpha': 1.819874390309593e-05, 'reg_lambda': 1.0175412298841808e-07}. Best is trial 0 with value: 0.5955685956657008.


...

[I 2026-03-20 15:15:26,074] Trial 47 finished with value: 0.5955685956657008 and parameters: {'max_depth': 3, 'n_estimators': 4800, 'learning_rate': 0.15417865724647117, 'min_child_weight': 7, 'gamma': 4.098249866902703, 'subsample': 0.5931437133093839, 'colsample_bytree': 0.7600677013309237, 'reg_alpha': 0.00011975149157375, 'reg_lambda': 0.010718823137163852}. Best is trial 0 with value: 0.5955685956657008.


[0]	validation_0-rmse:2.60587
[1]	validation_0-rmse:2.31108
[2]	validation_0-rmse:2.15219
[3]	validation_0-rmse:2.07163
[4]	validation_0-rmse:2.02273
[5]	validation_0-rmse:1.99900
[6]	validation_0-rmse:1.99001
[7]	validation_0-rmse:1.98057
[8]	validation_0-rmse:1.98459
[9]	validation_0-rmse:1.98631
[10]	validation_0-rmse:1.98279
[11]	validation_0-rmse:1.98039
[12]	validation_0-rmse:1.98315
[13]	validation_0-rmse:1.98357
[14]	validation_0-rmse:1.98491
[15]	validation_0-rmse:1.98553
[16]	validation_0-rmse:1.98514
[17]	validation_0-rmse:1.98524
[18]	validation_0-rmse:1.98522
[19]	validation_0-rmse:1.98488
[20]	validation_0-rmse:1.98572
[21]	validation_0-rmse:1.98186
[22]	validation_0-rmse:1.98311
[23]	validation_0-rmse:1.98358
[24]	validation_0-rmse:1.98593
[25]	validation_0-rmse:1.98598
[26]	validation_0-rmse:1.98688
[27]	validation_0-rmse:1.98729
[28]	validation_0-rmse:1.98636
[29]	validation_0-rmse:1.98904
[30]	validation_0-rmse:1.99021
[31]	validation_0-rmse:1.99089
[32]	validation_0-

[I 2026-03-20 15:15:26,390] Trial 48 finished with value: 0.5955685956657008 and parameters: {'max_depth': 14, 'n_estimators': 3600, 'learning_rate': 0.2461366285399894, 'min_child_weight': 11, 'gamma': 5.9042365086262825, 'subsample': 0.6600283272660039, 'colsample_bytree': 0.6954726420543413, 'reg_alpha': 0.30731779827359323, 'reg_lambda': 0.0018665064198204317}. Best is trial 0 with value: 0.5955685956657008.


[0]	validation_0-rmse:2.60587
[1]	validation_0-rmse:2.31108
[2]	validation_0-rmse:2.15219
[3]	validation_0-rmse:2.07163
[4]	validation_0-rmse:2.02273
[5]	validation_0-rmse:1.99900
[6]	validation_0-rmse:1.99001
[7]	validation_0-rmse:1.98057
[8]	validation_0-rmse:1.98459
[9]	validation_0-rmse:1.98631
[10]	validation_0-rmse:1.98279
[11]	validation_0-rmse:1.98039
[12]	validation_0-rmse:1.98315
[13]	validation_0-rmse:1.98357
[14]	validation_0-rmse:1.98491
[15]	validation_0-rmse:1.98553
[16]	validation_0-rmse:1.98514
[17]	validation_0-rmse:1.98524
[18]	validation_0-rmse:1.98522
[19]	validation_0-rmse:1.98488
[20]	validation_0-rmse:1.98572
[21]	validation_0-rmse:1.98186
[22]	validation_0-rmse:1.98311
[23]	validation_0-rmse:1.98358
[24]	validation_0-rmse:1.98593
[25]	validation_0-rmse:1.98598
[26]	validation_0-rmse:1.98688
[27]	validation_0-rmse:1.98729
[28]	validation_0-rmse:1.98636
[29]	validation_0-rmse:1.98904
[30]	validation_0-rmse:1.99021
[31]	validation_0-rmse:1.99089
[32]	validation_0-

[I 2026-03-20 15:15:26,733] Trial 49 finished with value: 0.5955685956657008 and parameters: {'max_depth': 7, 'n_estimators': 4000, 'learning_rate': 0.02445287083469634, 'min_child_weight': 17, 'gamma': 6.65424773326998, 'subsample': 0.8059469778440577, 'colsample_bytree': 0.7372237633294454, 'reg_alpha': 2.726266229822573e-06, 'reg_lambda': 0.2473048568404234}. Best is trial 0 with value: 0.5955685956657008.


In [95]:
study.best_params

{'max_depth': 11,
 'n_estimators': 4700,
 'learning_rate': 0.05532932798000539,
 'min_child_weight': 15,
 'gamma': 1.356098712183698,
 'subsample': 0.581094517772616,
 'colsample_bytree': 0.7659330310384236,
 'reg_alpha': 1.351853793308209,
 'reg_lambda': 0.36773855146634393}

In [96]:
study.best_value

0.5955685956657008

In [97]:
best_params = study.best_params

final_model = XGBRegressor(
    **best_params,
    objective="reg:squarederror",
    eval_metric="mae",
    tree_method="hist",
    verbosity=0,
    random_state=42
)

# train on FULL training data (train + valid)
final_model.fit(X, y)

# evaluate on untouched test set
from sklearn.metrics import r2_score

y_pred = final_model.predict(X_test)
print("Final Test R2:", r2_score(y_test, y_pred))

Final Test R2: 0.93601641852318


In [98]:
values = final_model.predict(test)

In [99]:
df = pd.DataFrame({
    'id':ids,
    'Age':values
})

In [100]:
df.head()

,id,Age
0,15000,14.295913
1,15001,13.036388
2,15002,10.097084
3,15003,10.623096
4,15004,8.399055


In [101]:
df.to_csv('sub_1.csv',index=False)

In [94]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
